# DeepResearcher Demo

A LangGraph-based research agent that decomposes a question into parallel research branches, fact-checks claims, and synthesizes a calibrated answer.

```
question
   │
   ▼
planner          → decomposes into N sub-questions
   │
   ▼ (Send API fan-out)
researcher × N   → parallel web search + claim extraction
   │
   ▼
fact_checker     → cross-compares claims for contradictions
   │
   ▼
synthesizer      → final answer with confidence score
```

In [1]:
import sys
sys.path.insert(0, "../src")

import uuid
from dotenv import load_dotenv
load_dotenv("../.env")

from deep_researcher.graph.builder import build_graph

graph = build_graph()
print("Graph built successfully")

Graph built successfully


## Run a query

Jupyter supports `await` directly in cells — no `asyncio.run()` needed.

In [2]:
QUESTION = "What are the best techniques for reducing hallucinations in large language models?"

result = await graph.ainvoke(
    {"question": QUESTION, "session_id": str(uuid.uuid4())},
    config={"configurable": {"thread_id": str(uuid.uuid4())}}
)

print("Done. Keys in result:", list(result.keys()))

Done. Keys in result: ['question', 'sub_questions', 'branch_results', 'contradictions', 'report', 'errors', 'session_id']


## Step 1: Planner output

The planner decomposes the question into focused sub-questions, each becoming an independent research branch.

In [4]:
for sq in result["sub_questions"]:
    print(f"[{sq.id}] {sq.question}")

[1] What are the common causes of hallucinations in large language models?
[2] What role does data quality play in the hallucination rates of language models?
[3] How can fine-tuning methods be employed to reduce hallucinations in language models?
[4] What are some successful case studies or research findings related to reducing hallucinations in large language models?
[5] How do interactive or feedback-based learning systems contribute to minimizing hallucinations in language models?


## Step 2: Researcher outputs

Each researcher ran in parallel. For each branch we can inspect the claims extracted and sources used.

In [5]:
for br in result["branch_results"]:
    print(f"\n--- Branch {br.sub_question_id} ---")
    print(f"Search queries: {br.search_queries_used}")
    print(f"Sources found: {len(br.sources)}")
    print(f"Claims extracted: {len(br.claims)}")
    for claim in br.claims:
        print(f"  [{claim.confidence:.2f}] {claim.text}")


--- Branch 1 ---
Search queries: ['What are the common causes of hallucinations in large language models?']
Sources found: 5
Claims extracted: 3
  [0.90] LLM hallucinations are instances in which an AI model confidently generates inaccurate outputs that aren’t justified by its training data.
  [0.85] AI hallucination can easily be missed by users, but understanding its various types can help identify the fabrications.
  [0.80] According to 'A Survey on Hallucination in Large Language Models' research paper, there are three types of LLM hallucination.

--- Branch 2 ---
Search queries: ['What role does data quality play in the hallucination rates of language models?']
Sources found: 5
Claims extracted: 3
  [0.90] OpenAI's research paper claims that language models hallucinate because standard training and evaluation procedures reward guessing over acknowledging uncertainty.
  [0.85] The paper suggests that adjustments to training and evaluation procedures could reduce hallucination rate

## Step 3: Fact checker output

The fact checker cross-compares claims across branches to surface contradictions.

In [6]:
contradictions = result["contradictions"]

if not contradictions:
    print("No contradictions found.")
else:
    for c in contradictions:
        print(f"[severity: {c.severity}] {c.explanation}")
        print(f"  claim A: {c.claim_a.text}")
        print(f"  claim B: {c.claim_b.text}")

No contradictions found.


## Step 4: Final report

In [7]:
from IPython.display import Markdown, display

report = result["report"]

display(Markdown(f"### Answer\n\n{report.answer}"))
print(f"\nConfidence: {report.confidence}")
print(f"Total claims: {len(report.claims)}")
print(f"Total sources: {len(report.sources)}")

if report.knowledge_gaps:
    print("\nKnowledge gaps:")
    for gap in report.knowledge_gaps:
        print(f"  • {gap}")

### Answer

The best techniques for reducing hallucinations in large language models (LLMs) involve a combination of training adjustments, advanced fine-tuning methods, and improved prompting strategies. Key techniques include:

1. **Improved Training Procedures**: Traditional training methods often encourage models to make confident guesses rather than acknowledge uncertainty. Adjusting these training protocols, as suggested by OpenAI, can significantly lower the chance of hallucinations. Promoting uncertainty can direct models to be more cautious when generating answers.

2. **Fine-Tuning with High-Quality Data**: Fine-tuning models on curated and contextually relevant datasets can help align outputs with factual information. Techniques such as Faithful Fine-Tuning (F2) employ specifically designed loss functions to enhance the faithfulness of the generated responses.

3. **Reinforcement Learning Approaches**: Using reinforcement-learning-based methods, such as those that utilize an Entity Hallucination Index (EHI) as a reward signal, targets entity-level hallucinations effectively. This structured feedback can refine model behavior toward accuracy.

4. **Decoding Strategies and Representation Editing**: Implementing decoding paradigms like Alternate Contrastive Decoding (ALCD) and representation editing can reduce misrepresentations and improve the alignment between the model's responses and the intended outputs without needing extensive retraining.

5. **Prompting Techniques**: Utilizing structured prompting methods, such as Chain-of-Thought (CoT) prompting, can guide models to reason through their outputs more logically and systematically, thereby minimizing inaccuracies. Techniques such as self-consistency decoding also aid in improving response reliability.

6. **Human Feedback**: Incorporating human feedback into the training process can provide models with contextually appropriate corrections that enhance their ability to generate factually correct responses.

In summary, a multifaceted approach that combines technological enhancements, procedural adjustments, and human oversight can effectively mitigate hallucinations in large language models.


Confidence: 0.85
Total claims: 16
Total sources: 25

Knowledge gaps:
  • Specific quantitative benchmarks for the effectiveness of various techniques in reducing hallucinations are not thoroughly detailed in the literature.
  • Long-term impact and sustainability of these techniques on model performance over time remain under-explored.
  • Further empirical studies comparing the efficacy of each technique across different tasks and domains would add clarity to their applicability.


## Configuration

All settings can be overridden via environment variables before building the graph.

| Variable | Default | Description |
|---|---|---|
| `LLM_MODEL` | `gpt-4o-mini` | Any LiteLLM-supported model |
| `MAX_SUB_QUESTIONS` | `5` | Number of parallel research branches |
| `SEARCH_PROVIDER` | `duckduckgo` | Use `tavily` for better results |
| `PAGE_EXTRACT_CHARS` | `2000` | Characters to extract per page |